[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Open-Athena/MarinFold/blob/main/notebooks/evals_exploration.ipynb)

# MarinFold evals exploration — [#250](https://github.com/Open-Athena/MarinFold/issues/250)

contacts-v1 predicts a protein's **residue–residue contact map from its sequence alone**, by
generating a document of `<contact> <pI> <pJ>` statements. Our published numbers for that are
aggregates — a mean R-precision over an eval set. This notebook is the per-protein view underneath
them.

Four parts, each usable on its own:

| | what it does | needs |
|---|---|---|
| **1. Scoreboard** | R-precision / AUC for every predictor on a chosen eval set, with bootstrap CIs | CPU |
| **2. Protein browser** | every protein × every predictor, joined to structural / homology / annotation features | CPU |
| **3. Contact maps** | run a checkpoint on one protein and plot its prediction against ground truth | **GPU** |
| **4. Two checkpoints** | the same protein under two models, side by side | **GPU** |

Everything is read from the public `open-athena/MarinFold` bucket — **no token, no cluster**.
Parts 3–4 need a GPU runtime (*Runtime → Change runtime type → T4*); the model is a 1.5B
checkpoint downloaded from the bucket.

### The two eval universes

They are kept separate on purpose and **must not be pooled or compared across**:

* **`foldbench-monomers`** — [#245](https://github.com/Open-Athena/MarinFold/issues/245)'s 333 FoldBench
  monomers, cut into `eval-val` (97 natural, the working set), `eval-test` (217 natural, held out —
  see the read budget below) and `eval-denovo` (19 de novo designs).
* **`legacy-554`** — the historical 554-protein set every published MarinFold number before #245
  lives on. 75 % de novo designed, and selected on for a year, so it answers "how does this compare
  to our earlier checkpoints" and nothing about generalisation.

They overlap in 112 stems but **disagree on the input sequence for 11 of them** (different chain
resolution), so a stem scored under one universe is not comparable to the same stem under the other.

In [ ]:
# @title Install (≈1 min) { display-mode: "form" }
# marinfold pins transformers<5 (the Levanter rope config is silently misread by 4.x otherwise
# — see MODELS.yaml). That pin also holds huggingface_hub < 1.0, so bucket files are read here
# over plain HTTPS resolve URLs rather than the hf_hub bucket API.
import importlib.util, subprocess, sys
from pathlib import Path

REPO = Path("/content/MarinFold") if Path("/content").exists() else Path.cwd().parent
if not (REPO / "marinfold" / "pyproject.toml").exists():
    subprocess.run(["git", "clone", "--depth", "1",
                    "https://github.com/Open-Athena/MarinFold.git", str(REPO)], check=True)
if importlib.util.find_spec("marinfold") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "-e", f"{REPO / 'marinfold'}[transformers]"], check=True)
print("repo:", REPO)

In [ ]:
# @title Configuration { run: "auto", display-mode: "form" }

# --- what to look at -----------------------------------------------------------------
UNIVERSE = "foldbench-monomers"  # @param ["foldbench-monomers", "legacy-554"]
EVAL_SET = "eval-val"  # @param ["eval-val", "eval-test", "eval-denovo", "natural (val+test)", "everything"]
RANGE = "all"  # @param ["all", "long", "medium", "short"]
METRIC = "R"  # @param ["R", "AUC", "L", "L/2", "L/5"]

# --- which protein to fold (parts 3-4) -----------------------------------------------
# Any stem in the selected universe. Part 2 prints the candidates; `1qys_A` (Top7, a de novo
# design) and `1ubq_A` live in legacy-554, not in the FoldBench monomer sets.
PROTEIN = "8ah9_A"  # @param {type:"string"}

MODEL = "contacts-v1-exp199-cooldown-1.5B"  # @param ["contacts-v1-exp199-cooldown-1.5B", "contacts-v1-exp199-1.5B", "contacts-v1-exp166-1.5B", "contacts-v1-exp117-1.5B", "contacts-v1-exp75-1.5B"]
MODEL_B = "contacts-v1-exp117-1.5B"  # @param ["contacts-v1-exp117-1.5B", "contacts-v1-exp75-1.5B", "contacts-v1-exp166-1.5B", "contacts-v1-exp199-1.5B", "contacts-v1-exp199-cooldown-1.5B"]

# --- inference recipe (exp82 settled values; change only deliberately) ----------------
N_ROLLOUTS = 100  # @param {type:"integer"}
TEMPERATURE = 1.0  # @param {type:"number"}
TOP_P = 0.95  # @param {type:"number"}
TOP_K = -1  # @param {type:"integer"}
BATCH_SIZE = 32  # @param {type:"integer"}

BASELINE = "ESMFold2"  # @param ["ESMFold2", "ESMFold", "Protenix-v2 single-seq", "Protenix-v2 + MSA", "seq-KNN (unfiltered corpus)"]
print({k: v for k, v in globals().items() if k.isupper() and not k.startswith("_")})

In [ ]:
# @title Load the published evals { display-mode: "form" }
import io, json, urllib.request
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pyarrow.parquet as pq

BUCKET = "https://huggingface.co/buckets/open-athena/MarinFold/resolve"
EXP245 = f"{BUCKET}/data/contacts-v1-foldbench-monomers-exp245"
EXP89 = f"{BUCKET}/data/contacts-v1-model-eval-exp89"
EXP247 = f"{BUCKET}/data/contacts-v1-protein-properties-exp247"
EXP199_COOLDOWN_ROWS = (
    f"{BUCKET}/data/contacts-v1-model-eval-exp199/replicates/cooldown-v2-20260815-01/derived/"
    "prot-exp199-cw-cv1-p06-cool-s01/step-290400/contact_eval_cw_p06_cool_step290400_rows.csv.gz"
)
# The legacy table names predictors by (model, mode); these are the rows to keep and what to call
# them. `distogram` rows are Protenix read out a second way and would double-count the predictor.
LEGACY_PREDICTORS = {
    ("marinfold-contacts-v1", "single_seq", "lm"): "MarinFold #75",
    ("esmfold", "single_seq", "structure"): "ESMFold",
    ("esmfold2", "single_seq", "structure"): "ESMFold2",
    ("protenix-v2", "single_seq", "structure"): "Protenix-v2 single-seq",
    ("protenix-v2", "msa", "structure"): "Protenix-v2 + MSA",
}


def fetch(url: str) -> bytes:
    """Read one public bucket file into memory (anonymous)."""
    with urllib.request.urlopen(url) as response:
        return response.read()


def load_universe(name: str) -> tuple[pd.DataFrame, dict]:
    """`(targets, ground_truth)` for one eval universe.

    `targets` is one row per scorable unit: dataset, stem, L, the exact input sequence the model
    is prompted with, and whatever annotation that universe carries. `ground_truth` maps
    `(dataset, stem)` to #89's record — resolved residues and pyconfind contacts with degrees.
    """
    if name == "foldbench-monomers":
        targets = pq.read_table(io.BytesIO(
            fetch(f"{EXP245}/eval_targets_foldbench_monomers.parquet"))).to_pandas()
        annotation = pd.read_csv(io.BytesIO(fetch(f"{EXP245}/eval_sets.csv")))
        targets = targets.merge(
            annotation[["stem", "eval_set", "designed", "is_viral", "kingdom", "title",
                        "deposit_date", "exp199_best_identity", "exp199_stratum"]],
            on="stem", how="left", validate="one_to_one")
        ground_truth_bytes = fetch(f"{EXP245}/gt_universe_scored.jsonl")
    elif name == "legacy-554":
        # The 554 input sequences are not on the bucket as a table; exp94's query FASTA is the
        # same set, verified byte-identical against the prompts exp89 actually used (554/554).
        rows = []
        fasta = (REPO / "experiments/exp94_evals_sequence_knn_baseline/data/eval_queries.fasta")
        for line in fasta.read_text().splitlines():
            if line.startswith(">"):
                dataset, stem = line[1:].strip().split("__", 1)
                rows.append({"dataset": dataset, "stem": stem, "input_seq": ""})
            elif line.strip():
                rows[-1]["input_seq"] += line.strip()
        targets = pd.DataFrame(rows)
        targets["L"] = targets.input_seq.str.len()
        targets["eval_set"] = "legacy-554"
        targets["designed"] = (targets.dataset == "denovo_pdb").astype(int)
        ground_truth_bytes = fetch(f"{EXP89}/gt_universe.jsonl")
    else:
        raise ValueError(f"unknown universe {name!r}")

    ground_truth = {}
    for line in ground_truth_bytes.decode().splitlines():
        record = json.loads(line)
        ground_truth[(record["dataset"], record["stem"])] = record
    missing = set(zip(targets.dataset, targets.stem)) - set(ground_truth)
    if missing:
        raise ValueError(f"{len(missing)} targets have no ground truth, e.g. {sorted(missing)[:3]}")
    return targets, ground_truth


def load_published(name: str) -> pd.DataFrame:
    """Published per-protein scores: `dataset, stem, predictor, range, cut, value`.

    Keyed by (dataset, stem), not stem: `7ur7_A` and `8ah9_A` are each in legacy-554 twice, under
    two datasets, with *different* input sequences and lengths. Joining on stem alone silently
    duplicates them and averages two different proteins together.
    """
    columns = ["dataset", "stem", "predictor", "range", "cut", "value"]
    if name == "foldbench-monomers":
        scores = pd.read_csv(io.BytesIO(fetch(f"{EXP245}/per_protein.csv.gz")), compression="gzip")
        # #245 scored one dataset only, and dropped the duplicate stems, so the key is safe to add.
        return scores.rename(columns={"precision": "value"}).assign(dataset="foldbench_monomer")[columns]

    legacy = pd.read_csv(io.BytesIO(fetch(f"{EXP89}/contact_precision_all.csv")))
    cooldown = pd.read_csv(io.BytesIO(fetch(EXP199_COOLDOWN_ROWS)), compression="gzip")
    cooldown = cooldown.assign(model="marinfold-contacts-v1", mode="single_seq")  # relabel below
    frames = []
    for frame, mapping in ((legacy, LEGACY_PREDICTORS),
                           (cooldown, {("marinfold-contacts-v1", "single_seq", "lm"):
                                       "MarinFold #199 cooldown"})):
        keys = list(zip(frame.model, frame["mode"], frame.predictor))
        frame = frame.assign(label=[mapping.get(key) for key in keys])
        frames.append(frame[frame.label.notna()])
    scores = pd.concat(frames, ignore_index=True)
    return (scores.rename(columns={"label": "predictor_name", "precision": "value"})
                  .drop(columns=["predictor"]).rename(columns={"predictor_name": "predictor"})
                  [columns])


def load_features() -> pd.DataFrame:
    """#247's 75 per-protein features (FoldBench monomers only)."""
    return pd.read_csv(io.BytesIO(fetch(f"{EXP247}/protein_features.csv")))


targets, ground_truth = load_universe(UNIVERSE)
published = load_published(UNIVERSE)
features = load_features()
# Some published tables cover a superset of the universe — the #199 cooldown was also scored on
# #226's 23 extra FoldBench chains, which are not part of the 554. Keep only this universe's units
# so a mean is over exactly the set it claims to be over.
in_universe = set(zip(targets.dataset, targets.stem))
outside = [key not in in_universe for key in zip(published.dataset, published.stem)]
if any(outside):
    dropped = published[outside]
    print(f"note: dropped {len(set(zip(dropped.dataset, dropped.stem)))} scored units that are not "
          f"in {UNIVERSE}")
    published = published[[not flag for flag in outside]]
print(f"{UNIVERSE}: {len(targets)} units, {published.predictor.nunique()} published predictors")
print(targets.eval_set.value_counts().to_string())

## 1. The scoreboard

Mean over proteins of the per-protein metric, with a percentile bootstrap CI over proteins.
Three rules from [#245](https://github.com/Open-Athena/MarinFold/issues/245) are enforced in the
output rather than left to the reader, because ignoring them changes the conclusion:

* **Designed and natural proteins are reported separately, never pooled.** Protenix-v2 single-seq
  scores 0.835 on designs and 0.265 on natural monomers; a pooled mean over a set that is 75 %
  designed (legacy-554) mostly reports how well a model folds idealised backbones.
* **Baseline comparisons need proteins that postdate the baselines' training cutoffs.** The
  FoldBench sets satisfy this by construction (0 of 217 eval-test units predate Protenix-v2's
  2021-09-30 cutoff). Half of legacy-554's designs do not, so a MarinFold-versus-baseline number
  there is contaminated *for the baselines* — compare our own checkpoints to each other and say so.
* **Differences under ~0.005 are ties** ([#204](https://github.com/Open-Athena/MarinFold/issues/204):
  four evaluations of one unchanged checkpoint span 0.0023).

**eval-test has a read budget.** It is a held-out confirmation set; selecting on it destroys it.
Every read gets logged in
[`experiments/exp245_evals_foldbench_held_out_monomers/data/eval_test_reads.md`](https://github.com/Open-Athena/MarinFold/blob/main/experiments/exp245_evals_foldbench_held_out_monomers/data/eval_test_reads.md).

In [ ]:
# @title Scoreboard { display-mode: "form" }
SET_FILTERS = {
    "eval-val": lambda frame: frame.eval_set == "eval-val",
    "eval-test": lambda frame: frame.eval_set == "eval-test",
    "eval-denovo": lambda frame: frame.eval_set == "eval-denovo",
    "natural (val+test)": lambda frame: frame.eval_set.isin(["eval-val", "eval-test"]),
    "everything": lambda frame: frame.stem.notna(),
}


def selected_units(frame: pd.DataFrame) -> pd.DataFrame:
    if UNIVERSE == "legacy-554":
        return frame
    return frame[SET_FILTERS[EVAL_SET](frame)]


def bootstrap_mean(values: np.ndarray, draws: int = 2_000, seed: int = 0) -> tuple[float, float, float]:
    values = values[~np.isnan(values)]
    if len(values) == 0:
        return np.nan, np.nan, np.nan
    rng = np.random.default_rng(seed)
    means = values[rng.integers(0, len(values), size=(draws, len(values)))].mean(axis=1)
    return float(values.mean()), float(np.percentile(means, 2.5)), float(np.percentile(means, 97.5))


def scoreboard(units: pd.DataFrame, scores: pd.DataFrame) -> pd.DataFrame:
    rows = []
    scores = scores[(scores.range == RANGE) & (scores.cut == METRIC)]
    for split, subset in (("designed", units[units.designed == 1]),
                          ("natural", units[units.designed == 0])):
        if subset.empty:
            continue
        keys = set(zip(subset.dataset, subset.stem))
        in_split = scores[[key in keys for key in zip(scores.dataset, scores.stem)]]
        for predictor, group in in_split.groupby("predictor"):
            mean, low, high = bootstrap_mean(group.value.values)
            rows.append(dict(split=split, n=len(group), predictor=predictor,
                             value=mean, ci_low=low, ci_high=high))
    return pd.DataFrame(rows).sort_values(["split", "value"], ascending=[True, False])


units = selected_units(targets)
if UNIVERSE == "foldbench-monomers" and EVAL_SET in ("eval-test", "natural (val+test)", "everything"):
    print("!! eval-test is a HELD-OUT set with a read budget. If this read informs a decision or a\n"
          "   published claim, append a row to exp245's data/eval_test_reads.md saying why.\n")

board = scoreboard(units, published)
print(f"{UNIVERSE} · {EVAL_SET if UNIVERSE == 'foldbench-monomers' else 'all 554'} · "
      f"{METRIC} ({RANGE}-range)\n")
print(board.to_string(index=False, float_format=lambda v: f"{v:.3f}"))

splits = [split for split in ("natural", "designed") if (board.split == split).any()]
fig, axes = plt.subplots(1, len(splits), figsize=(7.2 * len(splits), 4.4), squeeze=False)
for axis, split in zip(axes[0], splits):
    panel = board[board.split == split].sort_values("value")
    colors = ["#C44E52" if "MarinFold" in p or "#" in p else "#4C72B0" for p in panel.predictor]
    axis.barh(panel.predictor, panel.value, color=colors,
              xerr=[panel.value - panel.ci_low, panel.ci_high - panel.value],
              error_kw=dict(ecolor="0.3", lw=1.2, capsize=3))
    axis.set(xlabel=f"{METRIC} ({RANGE}-range)", xlim=(0, 1),
             title=f"{split} · n={int(panel.n.iloc[0])}")
    axis.grid(axis="x", alpha=0.3)
fig.suptitle(f"{UNIVERSE}{' · ' + EVAL_SET if UNIVERSE == 'foldbench-monomers' else ''}"
             f"  —  red = MarinFold checkpoints", y=1.02)
fig.tight_layout()
plt.show()

## 2. Protein browser

One row per protein: every predictor's score side by side, joined to
[#247](https://github.com/Open-Athena/MarinFold/issues/247)'s per-protein features (contact order,
secondary-structure content, MSA depth, best identity to the training corpus, annotations…).

Use it to pick a protein for parts 3–4 — the interesting ones are usually the extremes of
`delta` (MarinFold minus the chosen baseline), not the top of the list.

In [ ]:
# @title Per-protein table + scatter { display-mode: "form" }
MARINFOLD = {"foldbench-monomers": "#199 cooldown (contaminated)",
             "legacy-554": "MarinFold #199 cooldown"}[UNIVERSE]

wide = (published[(published.range == RANGE) & (published.cut == METRIC)]
        .pivot_table(index=["dataset", "stem"], columns="predictor", values="value")
        .reset_index())
columns = ["dataset", "stem", "L", "eval_set", "designed", "is_viral", "exp199_best_identity", "title"]
browser = (units[[c for c in columns if c in units.columns]]
           .merge(wide, on=["dataset", "stem"], how="inner", validate="one_to_one"))
if BASELINE in browser.columns and MARINFOLD in browser.columns:
    browser["delta"] = browser[MARINFOLD] - browser[BASELINE]
else:
    print(f"note: {BASELINE!r} was not scored on {UNIVERSE} — available predictors are "
          f"{sorted(published.predictor.unique())}\n")
feature_columns = ["stem", "relative_contact_order", "frac_helix", "frac_sheet",
                   "msa_log_depth", "knn_best_identity", "n_pfam"]
browser = browser.merge(features[[c for c in feature_columns if c in features.columns]],
                        on="stem", how="left", validate="many_to_one")
browser = browser.sort_values("delta" if "delta" in browser else MARINFOLD)

pd.set_option("display.width", 250, "display.max_columns", 60)
show = [c for c in ["dataset", "stem", "L", "eval_set", "designed", "is_viral", MARINFOLD, BASELINE, "delta",
                    "relative_contact_order", "frac_sheet", "msa_log_depth",
                    "exp199_best_identity"] if c in browser.columns]
print(f"--- worst 12 for MarinFold vs {BASELINE} ---")
print(browser[show].head(12).to_string(index=False, float_format=lambda v: f"{v:.3f}"))
print(f"\n--- best 12 ---")
print(browser[show].tail(12).iloc[::-1].to_string(index=False, float_format=lambda v: f"{v:.3f}"))

if "delta" in browser:
    fig, axis = plt.subplots(figsize=(6.4, 6.0))
    for label, subset, color in (("natural", browser[browser.designed == 0], "#4C72B0"),
                                 ("designed", browser[browser.designed == 1], "#DD8452")):
        axis.scatter(subset[BASELINE], subset[MARINFOLD], s=26, alpha=0.75, label=label, color=color)
    axis.plot([0, 1], [0, 1], color="0.4", lw=1, ls="--")
    axis.set(xlabel=f"{BASELINE}  ·  {METRIC} ({RANGE})", ylabel=f"MarinFold  ·  {METRIC} ({RANGE})",
             xlim=(0, 1.02), ylim=(0, 1.02),
             title=f"{UNIVERSE} · {EVAL_SET if UNIVERSE == 'foldbench-monomers' else 'legacy 554'}"
                   f"\nabove the line = MarinFold wins")
    axis.legend(); axis.grid(alpha=0.3)
    plt.show()

## 3. Predict a contact map

This runs the model. The recipe is the one [#82](https://github.com/Open-Athena/MarinFold/issues/82)
settled on and every published number since uses — **rollout + resample**:

1. Build `N_ROLLOUTS` *different* realizations of the same protein's document. contacts-v1 documents
   carry two nuisance randomizations (where the N-terminus lands in position-token space, and the
   order of the sequence statements), so each realization is a different prompt for the same protein.
2. Sample a contact-section completion from each (`temperature 1.0, top-p 0.95`, no top-k), with a
   generous token budget of `min(8192 − prompt, 6·L + 128)` so truncation is never the reason a
   document is short.
3. Count, for every residue pair, how many rollouts asserted it. That vote count is the score.
   Ties (mostly the large zero-vote mass) are broken by the pairwise log-probability readout.

Do not substitute the older pairwise-only readout for this: identical weights score ~0.086 lower
in R-precision under it, which reads like two generations of model progress.

**Metrics** use #89's implementation — candidate pairs are those between two *resolved* residues at
separation ≥ 6, a pair is a true contact at pyconfind degree ≥ 0.001, and R-precision is precision
in the top-R ranked pairs where R is that protein's true-contact count.

In [ ]:
# @title Rollout + metric implementation { display-mode: "form" }
from sklearn.metrics import roc_auc_score

from marinfold.document_structures.contacts_v1 import (
    InferenceConfig, predict, structure_from_sequence)

MIN_DEGREE, MIN_SEPARATION = 0.001, 6
RANGES = {"all": (6, None), "short": (6, 11), "medium": (12, 23), "long": (24, None)}
CUTS = (("L", lambda L, true: L), ("L/2", lambda L, true: max(1, L // 2)),
        ("L/5", lambda L, true: max(1, L // 5)), ("R", lambda L, true: true))


def true_matrix(length: int, contacts) -> np.ndarray:
    """#89's ground-truth contact matrix: degree >= 0.001, separation >= 6."""
    matrix = np.zeros((length, length), bool)
    for i, j, degree in contacts:
        i, j = int(i), int(j)
        if degree >= MIN_DEGREE and (j - i) >= MIN_SEPARATION and i < j < length:
            matrix[i, j] = matrix[j, i] = True
    return matrix


def candidate_pairs(resolved) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Upper-triangle pairs of resolved residues, and their sequence separations."""
    resolved = np.asarray(resolved)
    left, right = np.triu_indices(len(resolved), k=1)
    i, j = resolved[left], resolved[right]
    return i, j, j - i


def score_metrics(score: np.ndarray, record: dict) -> pd.DataFrame:
    """precision @ {L, L/2, L/5, R} + AUC per separation range — #89's metric_rows."""
    length = record["L"]
    truth = true_matrix(length, record["contacts"])
    i, j, separation = candidate_pairs(record["resolved"])
    pair_scores, pair_truth = score[i, j], truth[i, j].astype(int)
    rows = []
    for name, (low, high) in RANGES.items():
        in_range = separation >= low
        if high is not None:
            in_range &= separation <= high
        values, labels = pair_scores[in_range], pair_truth[in_range]
        n_candidate, n_true = int(values.size), int(labels.sum())
        ranked = labels[np.argsort(-values, kind="mergesort")] if n_candidate else None
        for cut, size_of in CUTS:
            target = int(size_of(length, n_true))
            top = min(target, n_candidate)
            rows.append(dict(range=name, cut=cut, n_candidate=n_candidate, n_true=n_true,
                             value=float(ranked[:top].sum()) / top if top > 0 else np.nan))
        auc = (float(roc_auc_score(labels, values))
               if n_candidate and 0 < n_true < n_candidate else np.nan)
        rows.append(dict(range=name, cut="AUC", n_candidate=n_candidate, n_true=n_true, value=auc))
    return pd.DataFrame(rows)


def resolve_target(name: str) -> pd.Series:
    """Look a protein up by `stem` or by the fully qualified `dataset__stem`."""
    if "__" in name:
        dataset, stem = name.split("__", 1)
        match = targets[(targets.dataset == dataset) & (targets.stem == stem)]
    else:
        match = targets[targets.stem == name]
    if match.empty:
        raise ValueError(f"{name!r} is not in {UNIVERSE}. Part 2 lists the {len(targets)} that are.")
    if len(match) > 1:
        # legacy-554 holds 7ur7_A and 8ah9_A twice, under two datasets, with different sequences.
        options = ", ".join(f"{d}__{s}" for d, s in zip(match.dataset, match.stem))
        raise ValueError(f"{name!r} is ambiguous in {UNIVERSE} — ask for one of: {options}")
    return match.iloc[0]


def fold(name: str, model: str, n_rollouts: int = None) -> dict:
    """Run one checkpoint on one protein and score it. Returns score matrix + metrics."""
    target = resolve_target(name)
    stem = target.stem
    config = InferenceConfig(
        model=model, backend="transformers", method="rollout", keep_matrix=True,
        n_rollouts=n_rollouts or N_ROLLOUTS, temperature=TEMPERATURE, top_p=TOP_P, top_k=TOP_K,
        batch_size=BATCH_SIZE, dtype="bfloat16", min_seq_separation=MIN_SEPARATION)
    record = next(iter(predict(config, structures=[
        structure_from_sequence(target.input_seq, entry_id=stem)])))
    # The band within MIN_SEPARATION comes back as NaN; it is never a candidate pair, but a NaN
    # would poison argsort, so push it below every real score.
    score = np.nan_to_num(np.asarray(record["score_matrix"], dtype=float), nan=-1e9)
    truth = ground_truth[(target.dataset, target.stem)]
    metrics = score_metrics(score, truth)
    return dict(stem=stem, model=model, target=target, score=score, truth=truth, metrics=metrics)


def headline(result: dict) -> str:
    metrics = result["metrics"]
    row = metrics[(metrics.range == "all") & (metrics.cut == "R")].iloc[0]
    auc = metrics[(metrics.range == "all") & (metrics.cut == "AUC")].value.iloc[0]
    long_r = metrics[(metrics.range == "long") & (metrics.cut == "R")].value.iloc[0]
    return (f"R-precision {row.value:.3f} (all) / {long_r:.3f} (long)   AUC {auc:.3f}   "
            f"[L={result['target'].L}, {int(row.n_true)} true contacts of "
            f"{int(row.n_candidate)} candidate pairs]")

In [ ]:
# @title Fold `PROTEIN` with `MODEL` { display-mode: "form" }
import time

start = time.time()
result = fold(PROTEIN, MODEL)
print(f"{MODEL} on {PROTEIN}  ({time.time() - start:.0f}s, {N_ROLLOUTS} rollouts)")
print(headline(result))

# How this compares to the published score for the same protein, where one exists. It will not
# match to the digit: the recipe is stochastic, this runs under transformers rather than the vLLM
# the eval harness uses, and the packaged recipe adds the tie-break. Same ballpark is the check.
target = result["target"]
match = published[(published.dataset == target.dataset) & (published.stem == target.stem)
                  & (published.range == "all") & (published.cut == "R")]
if not match.empty:
    print("\npublished R-precision (all) for this protein:")
    print(match[["predictor", "value"]].sort_values("value", ascending=False)
          .to_string(index=False, float_format=lambda v: f"{v:.3f}"))

In [ ]:
# @title Contact-map figure { display-mode: "form" }
from matplotlib.colors import LinearSegmentedColormap


def plot_contact_map(result: dict, axes=None, title: str = None):
    """Ground truth against prediction, twice: as a map, and as the top-R call."""
    score, truth_record = result["score"], result["truth"]
    length = truth_record["L"]
    truth = true_matrix(length, truth_record["contacts"])
    resolved = np.zeros(length, bool)
    resolved[np.asarray(truth_record["resolved"])] = True
    candidate = np.outer(resolved, resolved)
    candidate &= np.abs(np.subtract.outer(np.arange(length), np.arange(length))) >= MIN_SEPARATION

    # Panel 1: prediction above the diagonal, ground truth below it. Pairs the metric cannot
    # see — either endpoint unresolved in the deposited structure, or separation < 6 — are blanked
    # in both triangles, so the picture is exactly what the score is computed over.
    votes = np.where(candidate, score, np.nan)
    upper = np.triu(np.ones_like(votes, bool), k=1)
    panel = np.where(upper, votes, np.nan)
    panel_truth = np.where(~upper & truth, 1.0, np.nan)

    if axes is None:
        _, axes = plt.subplots(1, 2, figsize=(13.4, 6.4))
    heat = LinearSegmentedColormap.from_list("votes", ["#F2F2F2", "#F4C36B", "#C44E52", "#3B0A0C"])
    image = axes[0].imshow(panel, cmap=heat, origin="lower", interpolation="none",
                           vmin=0, vmax=max(1.0, np.nanmax(panel)))
    axes[0].imshow(panel_truth, cmap=LinearSegmentedColormap.from_list("gt", ["#333333", "#333333"]),
                   origin="lower", interpolation="none", vmin=0, vmax=1)
    axes[0].plot([0, length - 1], [0, length - 1], color="0.6", lw=0.8)
    plt.colorbar(image, ax=axes[0], fraction=0.046,
                 label=f"rollout votes (of {N_ROLLOUTS}, + tie-break)")
    axes[0].set(xlabel="residue j", ylabel="residue i",
                title=(title or f"{result['stem']} · {result['model']}") +
                      "\nupper: model votes   ·   lower: ground truth")

    # Panel 2: the top-R ranked pairs, right and wrong, over the ground truth.
    metrics = result["metrics"]
    n_true = int(metrics[(metrics.range == "all") & (metrics.cut == "R")].n_true.iloc[0])
    ranked = np.argsort(-np.where(candidate & upper, score, -np.inf), axis=None, kind="mergesort")
    top = np.unravel_index(ranked[:n_true], score.shape)
    hit = truth[top]
    axes[1].imshow(np.where(truth, 1.0, np.nan), cmap=LinearSegmentedColormap.from_list(
        "gt", ["#D8D8D8", "#D8D8D8"]), origin="lower", interpolation="none", vmin=0, vmax=1)
    for mask, color, label in ((hit, "#2A7F43", "correct"), (~hit, "#C44E52", "wrong")):
        axes[1].scatter(np.asarray(top[1])[mask], np.asarray(top[0])[mask], s=9, color=color,
                        label=f"{label} ({int(mask.sum())})")
        axes[1].scatter(np.asarray(top[0])[mask], np.asarray(top[1])[mask], s=9, color=color)
    axes[1].plot([0, length - 1], [0, length - 1], color="0.6", lw=0.8)
    axes[1].set(xlim=(-0.5, length - 0.5), ylim=(-0.5, length - 0.5),
                xlabel="residue j", ylabel="residue i",
                title=f"top-{n_true} predicted contacts over ground truth (grey)\n{headline(result)}")
    axes[1].legend(loc="lower right", fontsize=9)
    return axes


plot_contact_map(result)
plt.tight_layout()
plt.show()

## 4. Two checkpoints on the same protein

The same protein, folded twice. Useful for reading what a training change actually bought — the
aggregate says "+0.03 R-precision", the maps say whether it found a different fold or sharpened the
same one.

`MODEL_B` defaults to the [#117](https://github.com/Open-Athena/MarinFold/issues/117) sweep winner,
which was trained on AFDB alone; the default `MODEL` adds the ESM-Atlas half of the corpus and a
cooldown on top.

In [ ]:
# @title Fold with both checkpoints { display-mode: "form" }
result_b = fold(PROTEIN, MODEL_B)
print(f"{MODEL:<38} {headline(result)}")
print(f"{MODEL_B:<38} {headline(result_b)}")

figure, axes = plt.subplots(2, 2, figsize=(13.4, 12.4))
plot_contact_map(result, axes=axes[0], title=f"{PROTEIN} · {MODEL}")
plot_contact_map(result_b, axes=axes[1], title=f"{PROTEIN} · {MODEL_B}")
figure.tight_layout()
plt.show()

comparison = (result["metrics"].merge(result_b["metrics"], on=["range", "cut"],
                                      suffixes=("_a", "_b"))
              .assign(delta=lambda frame: frame.value_a - frame.value_b))
print(f"\n{MODEL} (a) vs {MODEL_B} (b)")
print(comparison[comparison.cut.isin(["R", "AUC"])][["range", "cut", "value_a", "value_b", "delta"]]
      .to_string(index=False, float_format=lambda v: f"{v:+.3f}"))

## What this notebook is not

It reads published artifacts and re-runs a recipe; it does not produce eval numbers of record.
Anything worth citing goes through
[exp245's harness](https://github.com/Open-Athena/MarinFold/tree/main/experiments/exp245_evals_foldbench_held_out_monomers)
and gets filed on the issue, for three reasons:

* **The recipe here is not bit-identical to the harness.** It runs under transformers rather than
  vLLM, and the packaged rollout adds a pairwise tie-break the eval worker does not apply. Both
  move a per-protein score by more than the aggregate noise floor.
* **Per-protein scores are noisy.** #204's four evaluations of one unchanged checkpoint agree to
  0.0023 *in aggregate over 554 proteins*; a single protein at 100 rollouts moves much more than
  that between runs.
* **A score is only as good as the set it is on.** A number read off one protein you chose after
  looking at the table is a selected number, and the eval-test read budget exists precisely because
  selection is invisible after the fact.

Two neighbours, so you pick the right notebook:
[`inference_example1.ipynb`](https://colab.research.google.com/github/Open-Athena/MarinFold/blob/main/notebooks/inference_example1.ipynb)
runs a checkpoint on **any RCSB entry** (its own ground truth, vLLM or transformers, no eval set),
and
[`fold_from_contacts1.ipynb`](https://colab.research.google.com/github/Open-Athena/MarinFold/blob/main/notebooks/fold_from_contacts1.ipynb)
turns predicted contacts into a **3D backbone**. This one is the one anchored to the eval universes
and the published per-protein scores.